In [1]:
import pandas as pd
import numpy as np

train = pd.read_csv(r'c:\Users\moham\OneDrive\Desktop\AI_memory_revival\MLOPS_pgp\data\processed\train.csv')
test  = pd.read_csv(r'c:\Users\moham\OneDrive\Desktop\AI_memory_revival\MLOPS_pgp\data\processed\test.csv')

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (79397, 29)
Test shape: (17014, 29)


In [2]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

In [3]:
print("=== Missing Values in Date Columns ===")
print(train[date_cols].isna().sum())

=== Missing Values in Date Columns ===
order_purchase_timestamp            0
order_approved_at                 110
order_delivered_carrier_date     1373
order_delivered_customer_date    2289
order_estimated_delivery_date       0
dtype: int64


In [4]:
for col in date_cols:
    train[col] = pd.to_datetime(train[col])
    test[col]  = pd.to_datetime(test[col])

print("✅ Date columns converted")
print(train[date_cols].dtypes)

✅ Date columns converted
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [5]:
train['is_late'] = (
    train['order_delivered_customer_date'] > train['order_estimated_delivery_date']
).astype(int)

# Mark missing delivery dates as NaN
missing_delivery = train['order_delivered_customer_date'].isna()
train.loc[missing_delivery, 'is_late'] = np.nan

print("=== is_late created ===")
print(train['is_late'].value_counts(dropna=False))

=== is_late created ===
is_late
0.0    70958
1.0     6150
NaN     2289
Name: count, dtype: int64


In [6]:
# See what order_status those missing rows have
print(train[missing_delivery]['order_status'].value_counts())
before = train.shape[0]
train = train.dropna(subset=['order_delivered_customer_date'])
after = train.shape[0]

print(f"\n✅ Undelivered orders dropped")
print(f"Removed: {before - after} rows")
print(f"Remaining rows: {after}")
print(f"Train shape after drop: {train.shape}")

order_status
shipped        864
canceled       498
unavailable    416
invoiced       253
processing     244
delivered        6
created          5
approved         3
Name: count, dtype: int64

✅ Undelivered orders dropped
Removed: 2289 rows
Remaining rows: 77108
Train shape after drop: (77108, 30)


In [16]:
counts = train['is_late'].value_counts()
pct = train['is_late'].value_counts(normalize=True) * 100
print(f"Late    (1): {counts[1]} rows  ({pct[1]:.1f}%)")
print(f"On time (0): {counts[0]} rows  ({pct[0]:.1f}%)")

Late    (1): 6150 rows  (8.0%)
On time (0): 70958 rows  (92.0%)


In [17]:
leaky_columns = [
    'order_delivered_customer_date',
    'order_delivered_carrier_date',
    'order_status'
]

print("=== LEAKY COLUMNS — NEVER USE AS FEATURES ===")
for col in leaky_columns:
    print(f"  don't use {col}")

=== LEAKY COLUMNS — NEVER USE AS FEATURES ===
  don't use order_delivered_customer_date
  don't use order_delivered_carrier_date
  don't use order_status
